In [ ]:
# Install and import required libraries
!pip install -q geopandas gdown --quiet

import geopandas as gpd
import gdown
import os

# Download the grid file from Google Drive
url = "https://drive.google.com/uc?id=1GMLbyT-iN6cqubGLOJX2rexzTmLNSbTD"
output = "grid_250m_with_nil_clean.geojson"
if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

# Load the grid GeoDataFrame
grid = gpd.read_file(output)
grid.crs


Downloading...
From: https://drive.google.com/uc?id=1GMLbyT-iN6cqubGLOJX2rexzTmLNSbTD
To: /content/grid_250m_with_nil_clean.geojson
100%|██████████| 1.44M/1.44M [00:00<00:00, 109MB/s]


<Projected CRS: EPSG:32632>
Name: WGS 84 / UTM zone 32N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 6°E and 12°E, northern hemisphere between equator and 84°N, onshore and offshore. Algeria. Austria. Cameroon. Denmark. Equatorial Guinea. France. Gabon. Germany. Italy. Libya. Liechtenstein. Monaco. Netherlands. Niger. Nigeria. Norway. Sao Tome and Principe. Svalbard. Sweden. Switzerland. Tunisia. Vatican City State.
- bounds: (6.0, 0.0, 12.0, 84.0)
Coordinate Operation:
- name: UTM zone 32N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
grid.head()

,grid_id,NIL,ID_NIL,geometry
0,0,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503447.317 5..."
1,1,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503212.574 5..."
2,2,ASSIANO,87.0,"POLYGON ((503447.317 5034431.613, 503447.317 5..."
3,3,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503447.317 5..."
4,4,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503402.364 5..."


In [ ]:
# Download and extract census sections shapefile
!gdown 1WpbI05lOC8hoOg-5Xitz_l2NGW3jnaRV --output R03_21.zip
!unzip -o R03_21.zip -d R03_21

import geopandas as gpd

# Read census sections shapefile
sezioni = gpd.read_file("R03_21/SHP/R03_21_WGS84.shp")

# Reproject to match the grid CRS (EPSG:32632)
sezioni = sezioni.to_crs(grid.crs)


Downloading...
From (original): https://drive.google.com/uc?id=1WpbI05lOC8hoOg-5Xitz_l2NGW3jnaRV
From (redirected): https://drive.google.com/uc?id=1WpbI05lOC8hoOg-5Xitz_l2NGW3jnaRV&confirm=t&uuid=59057d5b-ed44-42a9-b1dc-3509c4a33f0c
To: /content/R03_21.zip
100% 61.9M/61.9M [00:03<00:00, 15.7MB/s]
Archive:  R03_21.zip
   creating: R03_21/SHP/
  inflating: R03_21/SHP/R03_21_WGS84.dbf  
  inflating: R03_21/SHP/R03_21_WGS84.prj  
  inflating: R03_21/SHP/R03_21_WGS84.shp  
  inflating: R03_21/SHP/R03_21_WGS84.shx  
   creating: R03_21/TAB/
  inflating: R03_21/TAB/ASC1_R03_21.csv  
  inflating: R03_21/TAB/ASC2_R03_21.csv  
  inflating: R03_21/TAB/ISAM_R03_21.csv  
  inflating: R03_21/TAB/LOC_R03_21.csv  
  inflating: R03_21/TAB/R03_21.ods   
  inflating: R03_21/TAB/R03_21.xlsx  
  inflating: R03_21/TAB/SEZ_R03_21.csv  
  inflating: R03_21/TAB/ZIC_R03_21.csv  


In [ ]:
sezioni.head()

,COD_REG,COD_UTS,PRO_COM,SEZ21,SEZ21_ID,COD_TIPO_S,TIPO_LOC,LOC21_ID,COD_ZIC,COD_ISAM,...,COM_ASC1,COM_ASC2,COM_ASC3,POP21,FAM21,ABI21,EDI21,SHAPE_Leng,SHAPE_Area,geometry
0,3,12,12001,1,120010000001,1,1,1200110001,0,0,...,0,0,0,261,126,444,222,4980.782527,2.643492e+05,"POLYGON ((482412.527 5098154.95, 482393.468 50..."
1,3,12,12001,2,120010000002,1,1,1200110001,0,0,...,0,0,0,32,15,100,47,2624.994762,1.107193e+05,"POLYGON ((482339.494 5098223.769, 482332.623 5..."
2,3,12,12001,3,120010000003,1,2,1200126601,0,0,...,0,0,0,6,3,16,15,853.947919,2.264928e+04,"POLYGON ((481330.174 5097812.808, 481343.077 5..."
3,3,12,12001,7,120010000007,1,1,1200110002,0,0,...,0,0,0,1,1,5,4,1159.182228,1.388401e+04,"POLYGON ((482250.385 5097379.671, 482246.591 5..."
4,3,12,12001,10,120010000010,22,4,1200140001,0,0,...,0,0,0,73,31,64,43,12444.771483,1.413326e+06,"POLYGON ((482810.403 5097968.934, 482806.966 5..."


In [ ]:
sezioni_milano = sezioni[sezioni['PRO_COM'] == 15146].copy()


In [ ]:
import folium
import branca.colormap as cm
import numpy as np

# Assicuriamoci che sia in EPSG:4326
sezioni_milano = sezioni_milano.to_crs(epsg=4326)

# Minimo e massimo della popolazione (vmax = 95° percentile)
vmin = sezioni_milano["POP21"].min()
vmax = np.percentile(sezioni_milano["POP21"], 95)  # 95° percentile

# Colormap
colormap = cm.LinearColormap(
    colors=["white", "yellow", "orange", "red"],
    vmin=vmin,
    vmax=vmax,
    caption="Popolazione (2021)"
)

# Centro della mappa
center = sezioni_milano.geometry.unary_union.centroid
mappa = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

# Style function con clamp per i valori > vmax
def style_function(feature):
    value = feature["properties"]["POP21"]
    # Se maggiore di vmax, forza al massimo colore
    value_clamped = min(value, vmax) if value is not None else None
    return {
        "fillColor": colormap(value_clamped) if value_clamped is not None else "#cccccc",
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7
    }

# GeoJson con tooltip
folium.GeoJson(
    sezioni_milano[["SEZ21_ID", "POP21", "geometry"]],
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["SEZ21_ID", "POP21"],
        aliases=["Sezione:", "Popolazione:"],
        localize=True
    )
).add_to(mappa)

# Aggiungi colormap
colormap.add_to(mappa)

mappa


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
sezioni_prepped = sezioni_milano[['geometry', 'POP21']].copy()
sezioni_prepped = sezioni_prepped.reset_index().rename(columns={'index': 'sez_id'})
sezioni_prepped['A_s'] = sezioni_prepped.geometry.area

intersection = gpd.overlay(grid, sezioni_prepped[['sez_id', 'geometry', 'POP21']], how='intersection')


intersection['A_cs'] = intersection.geometry.area

intersection = intersection.merge(
    sezioni_prepped[['sez_id', 'A_s']],
    on='sez_id',
    how='left'
)

intersection['w_cs'] = intersection['A_cs'] / intersection['A_s']
intersection['p_cs'] = intersection['w_cs'] * intersection['POP21']

population_per_cell = intersection.groupby('grid_id', as_index=False)['p_cs'].sum()
population_per_cell = population_per_cell.rename(columns={'p_cs': 'population'})

In [ ]:
grid_with_pop = grid.merge(population_per_cell, on='grid_id', how='left')
grid_with_pop['population'] = grid_with_pop['population'].fillna(0)
grid_with_pop.head()

,grid_id,NIL,ID_NIL,geometry,population
0,0,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503447.317 5...",0.220602
1,1,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503212.574 5...",0.103777
2,2,ASSIANO,87.0,"POLYGON ((503447.317 5034431.613, 503447.317 5...",0.695188
3,3,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503447.317 5...",1.429449
4,4,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503402.364 5...",0.041181


In [ ]:
grid_with_pop[grid_with_pop['population'] > 0].head()


,grid_id,NIL,ID_NIL,geometry,population
0,0,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503447.317 5...",0.220602
1,1,ASSIANO,87.0,"POLYGON ((503447.317 5032681.613, 503212.574 5...",0.103777
2,2,ASSIANO,87.0,"POLYGON ((503447.317 5034431.613, 503447.317 5...",0.695188
3,3,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503447.317 5...",1.429449
4,4,ASSIANO,87.0,"POLYGON ((503447.317 5034681.613, 503402.364 5...",0.041181


In [ ]:
import folium
import branca.colormap as cm
import numpy as np

# Assicuriamoci di avere CRS WGS84
grid_with_pop = grid_with_pop.to_crs(epsg=4326)

# Calcolo vmin e vmax (95° percentile)
vmin = grid_with_pop["population"].min()
vmax = np.percentile(grid_with_pop["population"], 95)  # 95° percentile

# Colormap coerente con le sezioni di censimento
colormap = cm.LinearColormap(
    colors=["white", "yellow", "orange", "red"],  # stessa palette delle sezioni
    vmin=vmin,
    vmax=vmax,
    caption="Popolazione stimata (2021) per cella 250m"
)

# Centro mappa
center = grid_with_pop.geometry.unary_union.centroid
mappa = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="cartodbpositron")

# Style function con clamp > vmax
def style_function(feature):
    value = feature["properties"]["population"]
    value_clamped = min(value, vmax) if value is not None else None
    return {
        "fillColor": colormap(value_clamped) if value_clamped is not None else "#cccccc",
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7
    }

# GeoJson con tooltip
folium.GeoJson(
    grid_with_pop[["grid_id", "population", "geometry"]],
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["grid_id", "population"],
        aliases=["Cella 250m:", "Popolazione stimata:"],
        localize=True
    )
).add_to(mappa)

# Aggiungi colormap
colormap.add_to(mappa)

mappa


/tmp/ipython-input-1038722386.py:21: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = grid_with_pop.geometry.unary_union.centroid


In [ ]:
pop_celle = grid_with_pop['population'].sum()
pop_sezioni = sezioni_prepped['POP21'].sum()
diff = pop_celle - pop_sezioni

print(f"Totale popolazione da celle: {pop_celle:,.0f}")
print(f"Totale popolazione da sezioni: {pop_sezioni:,.0f}")
print(f"Differenza assoluta: {diff:.4f}")
print(f"Errore relativo: {100 * abs(diff) / pop_sezioni:.6f}%")


Totale popolazione da celle: 1,349,494
Totale popolazione da sezioni: 1,349,930
Differenza assoluta: -435.7105
Errore relativo: 0.032277%


In [ ]:
grid_with_pop['population'].mean()

np.float64(434.6197389711432)

In [ ]:
percentili = grid_with_pop['population'].quantile([i/100 for i in range(1, 50)])
print(percentili)


0.01      0.000000
0.02      0.000000
0.03      0.000000
0.04      0.000000
0.05      0.000000
0.06      0.000000
0.07      0.000000
0.08      0.000000
0.09      0.000000
0.10      0.015738
0.11      0.056461
0.12      0.123040
0.13      0.193796
0.14      0.270441
0.15      0.345141
0.16      0.434724
0.17      0.529293
0.18      0.672385
0.19      0.795705
0.20      0.982289
0.21      1.242518
0.22      1.491352
0.23      1.835320
0.24      2.106604
0.25      2.824436
0.26      3.401960
0.27      4.500017
0.28      5.216402
0.29      6.283554
0.30      7.629322
0.31      9.897029
0.32     13.614891
0.33     16.786175
0.34     20.956954
0.35     27.483404
0.36     34.394777
0.37     40.479325
0.38     46.902464
0.39     51.936750
0.40     60.480454
0.41     75.499124
0.42     86.196806
0.43     97.058461
0.44    109.485406
0.45    126.156160
0.46    141.926104
0.47    158.974197
0.48    179.102560
0.49    197.954026
Name: population, dtype: float64


In [ ]:
grid_with_pop.to_file("grid_with_population.gpkg", layer="grid", driver="GPKG")
